# 02 — Feature engineering and the split

Two decisions: how the categorical codes are encoded, and how the data is divided.
Both are made once, here, and every later notebook inherits them.

In [1]:
import sys; sys.path.insert(0, '..')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid')
EXP = '../experiments/'

from src.data import load_uci, duplicate_report, TARGET, PAY_COLS
from src.features import (PAY_LEVELS, EDUCATION_MAP, MARRIAGE_MAP, collapse_codes,
                          build_preprocessor, CATEGORICAL, NUMERIC)
from src.split import make_splits, load_split, split_summary, why_not_temporal

df = load_uci()

## Collapsing the codes

Three collapses, all **stateless** — fixed maps, nothing estimated from data. A stateless
transform cannot carry information across a split, so it is leak-safe anywhere. Anything that
does learn from data (the scaler) lives inside the sklearn pipeline, fit on train only.

**`PAY_*` → categories, with everything ≥ 3 pooled.** Notebook 01 showed the codes are not
ordered. Separately, the high codes are thin.

In [2]:
counts = df[PAY_COLS[0]].value_counts().sort_index()
rates = df.groupby(PAY_COLS[0])[TARGET].mean()
thin = pd.DataFrame({'n': counts, 'default_rate': rates}).loc[3:]
display(thin.round(4))
print(f"codes >= 3 together: {thin.n.sum():,} rows ({thin.n.sum()/len(df):.2%} of the data), "
      f"and their rates are not monotone: {list(thin.default_rate.round(3))}")
print(f"collapsed levels: {PAY_LEVELS}")

,n,default_rate
PAY_0,,
3,322,0.7578
4,76,0.6842
5,26,0.5000
6,11,0.5455
7,9,0.7778
8,19,0.5789


codes >= 3 together: 463 rows (1.54% of the data), and their rates are not monotone: [0.758, 0.684, 0.5, 0.545, 0.778, 0.579]
collapsed levels: ['-2', '-1', '0', '1', '2', '3+']


**`EDUCATION` and `MARRIAGE` → named levels.** Codes 0, 5, 6 of `EDUCATION` and code 0 of
`MARRIAGE` are undocumented; code 4 and code 3 are "others". Pooled, they are small enough
that separate cells would be fitting noise.

In [3]:
print('EDUCATION:', EDUCATION_MAP)
print('MARRIAGE: ', MARRIAGE_MAP)
for col, mapping in [('EDUCATION', EDUCATION_MAP), ('MARRIAGE', MARRIAGE_MAP)]:
    g = df.assign(level=df[col].map(mapping).fillna('other')).groupby('level').agg(
        n=(TARGET, 'size'), default_rate=(TARGET, 'mean'))
    print(f"--- {col} after collapsing ---"); display(g.round(4))

EDUCATION: {1: 'graduate', 2: 'university', 3: 'high_school', 0: 'other', 4: 'other', 5: 'other', 6: 'other'}
MARRIAGE:  {1: 'married', 2: 'single', 0: 'other', 3: 'other'}
--- EDUCATION after collapsing ---


,n,default_rate
level,,
graduate,10585,0.1923
high_school,4917,0.2516
other,468,0.0705
university,14030,0.2373


--- MARRIAGE after collapsing ---


,n,default_rate
level,,
married,13659,0.2347
other,377,0.2361
single,15964,0.2093


In [4]:
before = df[CATEGORICAL].head(3)
after = collapse_codes(df).head(3)[CATEGORICAL]
print('before:'); display(before)
print('after:'); display(after)

before:


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,EDUCATION,MARRIAGE,SEX
ID,,,,,,,,,
1,2,2,-1,-1,-2,-2,2,1,2
2,-1,2,0,0,0,2,2,2,2
3,0,0,0,0,0,0,2,2,2


after:


,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,EDUCATION,MARRIAGE,SEX
ID,,,,,,,,,
1,2,2,-1,-1,-2,-2,university,married,female
2,-1,2,0,0,0,2,university,single,female
3,0,0,0,0,0,0,university,single,female


## One-hot encoding, inside a pipeline

`build_preprocessor` one-hot encodes the categoricals and (for linear models only) scales the
numerics. It is a `ColumnTransformer` inside each model's `Pipeline`, so cross-validation
re-fits it per fold. That is the structural defence against preprocessing leakage: the scaler
never sees the fold it is scoring.

`handle_unknown='infrequent_if_exist'` with `min_frequency=20` means a category too rare to
have been learned does not raise when it turns up later.

In [5]:
prep = build_preprocessor(scale=True)
Xt = prep.fit_transform(collapse_codes(df.drop(columns=[TARGET])))
names = list(prep.get_feature_names_out())
print(f"{df.shape[1] - 1} raw columns -> {Xt.shape[1]} encoded features")
print(f"categorical: {CATEGORICAL}")
print(f"numeric:     {NUMERIC}")
print('first 15 feature names:', names[:15])

23 raw columns -> 57 encoded features
categorical: ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'EDUCATION', 'MARRIAGE', 'SEX']
numeric:     ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
first 15 feature names: ['PAY_0_-1', 'PAY_0_-2', 'PAY_0_0', 'PAY_0_1', 'PAY_0_2', 'PAY_0_3+', 'PAY_2_-1', 'PAY_2_-2', 'PAY_2_0', 'PAY_2_1', 'PAY_2_2', 'PAY_2_3+', 'PAY_3_-1', 'PAY_3_-2', 'PAY_3_0']


## Deduplicate, then split

Order matters. Some rows in UCI are exact duplicates of another row on every feature. If a row
and its twin land on opposite sides of the split, the model is scored on data it memorised.
Deduplicating *after* splitting would leave those cross-split pairs intact.

In [6]:
feature_cols = [c for c in df.columns if c != TARGET]
rep = duplicate_report(df, feature_cols)
print(rep)
print(f"{rep['n_redundant_rows']} rows are redundant copies, in {rep['n_groups']} duplicate groups")

{'n_rows': 30000, 'n_rows_in_dup_groups': 108, 'n_redundant_rows': 56, 'n_groups': 52}
56 rows are redundant copies, in 52 duplicate groups


Then a stratified 60 / 20 / 20 split. Stratified because at a 22% base rate an unstratified
20% slice can easily drift a point or two on class balance, which moves every metric that
depends on the base rate.

- **train** — fit models
- **validation** — model selection, hyperparameters, the cutoff
- **test** — scored once, at the end

In [7]:
make_splits()
summary = split_summary()
display(summary)
rep = duplicate_report(load_uci(), [c for c in load_uci().columns if c != TARGET])
summary.assign(duplicate_rows_dropped=rep['n_redundant_rows']).to_csv(EXP + 'split_summary.csv')
print('default rate is within a rounding error across all three splits')

,n,share,default_rate,sha256_16
train,17966,0.6,0.22114,342216416726495d
val,5989,0.2,0.221239,3e8715486444df85
test,5989,0.2,0.221072,b82c7669f9bc586c


default rate is within a rounding error across all three splits


### No out-of-time split

Worth being explicit about, because an out-of-time split is what a bank would actually do.

In [8]:
print(why_not_temporal())

Why this dataset cannot support a temporal split, stated explicitly.

    A temporal split needs rows that can be ordered in time. Here every customer
    is observed over the *same* calendar window -- April to September 2005 -- with
    one row each and a single shared outcome month, October 2005. There is no
    column that orders customers in time, no cohort structure, and no second
    snapshot of anyone.

    For a real behaviour scorecard a temporal (out-of-time) split is the correct
    design: you train on older cohorts and validate on newer ones, because it is
    the only way to measure whether the model survives population drift. That is
    what a bank would do. We cannot, and inventing a proxy -- ordering by ID, say
    -- would fabricate a validation property the data does not have.

    Consequence to state in the write-up: these estimates measure generalisation
    to *unseen customers*, not to a *future time period*.
    
